In [ ]:
"""
F1 Live Data Pipeline — DS591 Boston University
=================================================
TWO MODES:
  MODE 1 (Tonight / fast): Direct push from FastF1 → Power BI REST URL
  MODE 2 (Full Azure):     FastF1 → Azure Event Hub → Stream Analytics → Power BI

Architecture:
  Thread 1 — record_live_session(): SignalRClient streams raw F1 data to .txt file
  Thread 2 — poll_and_push():       Reloads file every POLL_INTERVAL seconds,
                                    extracts lap/result data, sends to Power BI or Event Hub

Authentication:
  Set environment variables BEFORE running:
    export F1_USERNAME="your_formula1.com_email"
    export F1_PASSWORD="your_formula1.com_password"
    export POWER_BI_PUSH_URL="https://api.powerbi.com/beta/..."   # Mode 1
    export EVENT_HUB_CONNECTION_STRING="Endpoint=sb://..."         # Mode 2
    export EVENT_HUB_NAME="your-hub-name"                          # Mode 2

Azure Deployment:
  Add ALL env vars above as Azure App Service → Configuration → Application Settings.
  Never hardcode credentials.
"""

import json
import os
import time
import threading
import logging
from datetime import datetime, timezone
from pathlib import Path

import fastf1
from fastf1.livetiming.client import SignalRClient
from fastf1.livetiming.data import LiveTimingData
import requests

# ── Try importing Azure Event Hub SDK (optional — only needed for Mode 2) ──────
try:
    from azure.eventhub import EventHubProducerClient, EventData
    AZURE_SDK_AVAILABLE = True
except ImportError:
    AZURE_SDK_AVAILABLE = False

# ══════════════════════════════════════════════════════════════════════
# LOGGING
# ══════════════════════════════════════════════════════════════════════
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("f1_pipeline.log"),
    ],
)
log = logging.getLogger("F1Pipeline")

# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit these or set as environment variables
# ══════════════════════════════════════════════════════════════════════
YEAR         = 2025
GRAND_PRIX   = "Japan"   # FastF1 GP name: "Japan", "Bahrain", "Monaco", etc.
SESSION_TYPE = "R"       # R=Race, Q=Qualifying, FP1/FP2/FP3=Practice

LIVE_DATA_FILE = "live_timing_data.txt"
POLL_INTERVAL  = 10      # seconds between polls — keep ≥10 to avoid file-write conflicts

# ── MODE 1: Direct Power BI push ──────────────────────────────────────
# Get this from: Power BI → your streaming dataset → "API info" → Push URL
POWER_BI_PUSH_URL = os.environ.get(
    "POWER_BI_PUSH_URL",
    "https://api.powerbi.com/beta/YOUR_PUSH_DATASET_URL"   # ← replace or set env var
)

# ── MODE 2: Azure Event Hub ───────────────────────────────────────────
# Get connection string from: Azure Portal → Event Hub Namespace → Shared Access Policies
EVENT_HUB_CONNECTION_STRING = os.environ.get("EVENT_HUB_CONNECTION_STRING", "")
EVENT_HUB_NAME              = os.environ.get("EVENT_HUB_NAME", "f1-live-data")

# Decide which mode to use automatically
USE_EVENT_HUB = (
    AZURE_SDK_AVAILABLE
    and bool(EVENT_HUB_CONNECTION_STRING)
    and "YOUR_PUSH_DATASET_URL" in POWER_BI_PUSH_URL   # fallback if PBI not configured
)

fastf1.Cache.enable_cache("./cache")


# ══════════════════════════════════════════════════════════════════════
# HELPER — safe JSON serialisation for pandas/numpy types
# ══════════════════════════════════════════════════════════════════════
def safe(val):
    """Convert pandas/numpy types to JSON-serialisable Python primitives."""
    if val is None:
        return None
    try:
        import pandas as pd
        import numpy as np
        if pd.isna(val):
            return None
        if isinstance(val, np.integer):
            return int(val)
        if isinstance(val, np.floating):
            return float(val)
        if isinstance(val, pd.Timedelta):
            return round(val.total_seconds(), 4)
        if isinstance(val, pd.Timestamp):
            return val.isoformat()
        return val
    except Exception:
        return str(val)


# ══════════════════════════════════════════════════════════════════════
# DATA EXTRACTION
# ══════════════════════════════════════════════════════════════════════
def extract_laps_data(session) -> list[dict]:
    """
    Extract ALL lap records from the session.
    Returns every lap for every driver (not just the latest per driver).
    We track what we've already pushed using a seen-set of (driver, lap_number).
    """
    try:
        laps = session.laps
        if laps is None or laps.empty:
            log.debug("No lap data available yet.")
            return []

        output = []
        for _, row in laps.iterrows():
            record = {
                # Metadata
                "timestamp_utc":    datetime.now(timezone.utc).isoformat(),
                "year":             YEAR,
                "grand_prix":       GRAND_PRIX,
                "session_type":     SESSION_TYPE,
                # Driver
                "driver_code":      safe(row.get("Driver")),
                "driver_number":    safe(row.get("DriverNumber")),
                "team":             safe(row.get("Team")),
                # Lap timing (seconds)
                "lap_time_s":       safe(row.get("LapTime")),
                "sector1_time_s":   safe(row.get("Sector1Time")),
                "sector2_time_s":   safe(row.get("Sector2Time")),
                "sector3_time_s":   safe(row.get("Sector3Time")),
                # Position & lap info
                "position":         safe(row.get("Position")),
                "lap_number":       safe(row.get("LapNumber")),
                "stint":            safe(row.get("Stint")),
                # Tyre
                "compound":         safe(row.get("Compound")),
                "tyre_life":        safe(row.get("TyreLife")),
                "fresh_tyre":       safe(row.get("FreshTyre")),
                # Speed traps (km/h)
                "speed_i1":         safe(row.get("SpeedI1")),
                "speed_i2":         safe(row.get("SpeedI2")),
                "speed_fl":         safe(row.get("SpeedFL")),
                "speed_st":         safe(row.get("SpeedST")),
                # Validity flags
                "is_personal_best": safe(row.get("IsPersonalBest")),
                "deleted":          safe(row.get("Deleted")),
                "deleted_reason":   safe(row.get("DeletedReason")),
                "is_accurate":      safe(row.get("IsAccurate")),
                # Pit stops
                "pit_out_time_s":   safe(row.get("PitOutTime")),
                "pit_in_time_s":    safe(row.get("PitInTime")),
                # Session timestamps
                "lap_start_date":   safe(row.get("LapStartDate")),
            }
            output.append(record)
        return output

    except Exception as e:
        log.error(f"Error extracting lap data: {e}", exc_info=True)
        return []


def extract_results_data(session) -> list[dict]:
    """Extract session results / classification (available after session ends or Q segments)."""
    try:
        results = session.results
        if results is None or results.empty:
            return []

        output = []
        for _, row in results.iterrows():
            record = {
                "timestamp_utc":          datetime.now(timezone.utc).isoformat(),
                "grand_prix":             GRAND_PRIX,
                "session_type":           SESSION_TYPE,
                "driver_number":          safe(row.get("DriverNumber")),
                "driver_code":            safe(row.get("Abbreviation")),
                "full_name":              safe(row.get("FullName")),
                "team_name":              safe(row.get("TeamName")),
                "grid_position":          safe(row.get("GridPosition")),
                "classified_position":    safe(row.get("ClassifiedPosition")),
                "position":               safe(row.get("Position")),
                "q1_time_s":              safe(row.get("Q1")),
                "q2_time_s":              safe(row.get("Q2")),
                "q3_time_s":              safe(row.get("Q3")),
                "time_s":                 safe(row.get("Time")),
                "status":                 safe(row.get("Status")),
                "points":                 safe(row.get("Points")),
            }
            output.append(record)
        return output

    except Exception as e:
        log.error(f"Error extracting results: {e}", exc_info=True)
        return []


# ══════════════════════════════════════════════════════════════════════
# OUTPUT — MODE 1: Direct Power BI push
# ══════════════════════════════════════════════════════════════════════
def push_to_powerbi(data: list[dict], label: str = "laps"):
    """POST records to a Power BI streaming dataset push URL."""
    if not data:
        return

    if "YOUR_PUSH_DATASET_URL" in POWER_BI_PUSH_URL:
        log.warning(f"[DRY RUN] POWER_BI_PUSH_URL not set. Would push {len(data)} {label} records.")
        log.debug(json.dumps(data[:2], indent=2, default=str))
        return

    try:
        resp = requests.post(
            POWER_BI_PUSH_URL,
            headers={"Content-Type": "application/json"},
            data=json.dumps(data, default=str),
            timeout=15,
        )
        if resp.status_code in (200, 201, 202):
            log.info(f"✅ Power BI: pushed {len(data)} {label} records.")
        else:
            log.error(f"❌ Power BI push failed [{resp.status_code}]: {resp.text[:200]}")
    except requests.exceptions.Timeout:
        log.warning("Power BI push timed out — will retry next poll.")
    except Exception as e:
        log.error(f"Power BI push error: {e}", exc_info=True)


# ══════════════════════════════════════════════════════════════════════
# OUTPUT — MODE 2: Azure Event Hub
# ══════════════════════════════════════════════════════════════════════
def send_to_event_hub(data: list[dict], label: str = "laps"):
    """
    Send records to Azure Event Hub as individual EventData messages.
    Event Hub → Stream Analytics Job → Power BI streaming dataset.
    """
    if not data or not AZURE_SDK_AVAILABLE or not EVENT_HUB_CONNECTION_STRING:
        log.warning("Event Hub not configured or azure-eventhub not installed.")
        return

    try:
        producer = EventHubProducerClient.from_connection_string(
            conn_str=EVENT_HUB_CONNECTION_STRING,
            eventhub_name=EVENT_HUB_NAME,
        )
        with producer:
            event_data_batch = producer.create_batch()
            for record in data:
                msg = json.dumps(record, default=str).encode("utf-8")
                try:
                    event_data_batch.add(EventData(msg))
                except ValueError:
                    # Batch full — send current batch and start a new one
                    producer.send_batch(event_data_batch)
                    log.info(f"  → Sent partial batch to Event Hub")
                    event_data_batch = producer.create_batch()
                    event_data_batch.add(EventData(msg))
            producer.send_batch(event_data_batch)
        log.info(f"✅ Event Hub: sent {len(data)} {label} records → {EVENT_HUB_NAME}")
    except Exception as e:
        log.error(f"Event Hub error: {e}", exc_info=True)


def output_data(data: list[dict], label: str = "laps"):
    """Route data to Event Hub (Mode 2) or Power BI direct push (Mode 1)."""
    if USE_EVENT_HUB:
        send_to_event_hub(data, label)
    else:
        push_to_powerbi(data, label)


# ══════════════════════════════════════════════════════════════════════
# THREAD 1: Live recording (SignalR → .txt file)
# ══════════════════════════════════════════════════════════════════════
def record_live_session():
    """
    Stream F1 live timing data to disk via SignalRClient.
    Handles reconnection automatically (SignalR has a 2-hour session limit).
    Blocks until session ends or Ctrl+C.

    Auth: FastF1 reads F1_USERNAME + F1_PASSWORD env vars automatically.
    """
    session_part = 0
    while True:
        session_part += 1
        filename = LIVE_DATA_FILE if session_part == 1 else f"live_timing_data_part{session_part}.txt"

        log.info(f"🎙️  SignalRClient → {filename} (part {session_part})")

        try:
            client = SignalRClient(
                filename=filename,
                filemode="w",
                timeout=300,    # Stop if no data for 5 min (session gap / end)
                debug=False,
                no_auth=False,  # Reads F1_USERNAME + F1_PASSWORD from env
            )
            client.start()      # BLOCKS here
            log.info("SignalRClient stopped — session likely ended.")
            break
        except KeyboardInterrupt:
            log.info("Recording stopped by user (Ctrl+C).")
            break
        except Exception as e:
            log.error(f"SignalRClient error: {e}. Reconnecting in 10s…", exc_info=True)
            time.sleep(10)


# ══════════════════════════════════════════════════════════════════════
# THREAD 2: Poll & push (read .txt → extract → push)
# ══════════════════════════════════════════════════════════════════════
def poll_and_push():
    """
    Every POLL_INTERVAL seconds:
    1. Load all live_timing_data*.txt files via LiveTimingData
    2. Parse with FastF1 session.load()
    3. Push only NEW records since last poll (deduplication via seen set)
    """
    log.info("🔄 Poll loop started. Waiting for data file…")

    # Deduplication: track (driver_code, lap_number) pairs already pushed
    seen_laps: set[tuple] = set()
    seen_results: set[str] = set()   # driver_code

    while True:
        time.sleep(POLL_INTERVAL)

        try:
            # Collect all data file parts (handles SignalR reconnections)
            data_files = sorted(Path(".").glob("live_timing_data*.txt"))
            if not data_files:
                log.info("No data files yet — waiting…")
                continue

            file_paths = [str(f) for f in data_files]
            log.debug(f"Loading: {file_paths}")

            livedata = LiveTimingData(*file_paths)

            session = fastf1.get_session(YEAR, GRAND_PRIX, SESSION_TYPE)
            session.load(
                livedata=livedata,
                laps=True,
                telemetry=False,    # Disable — too large for live streaming
                weather=True,
                messages=True,
            )

            # ── Laps ──────────────────────────────────────────────────
            all_laps = extract_laps_data(session)
            new_laps = []
            for rec in all_laps:
                key = (rec.get("driver_code"), rec.get("lap_number"))
                if key not in seen_laps and rec.get("lap_number") is not None:
                    seen_laps.add(key)
                    new_laps.append(rec)

            if new_laps:
                output_data(new_laps, "laps")
                log.info(f"📊 {len(new_laps)} new laps pushed. Total seen: {len(seen_laps)}")
            else:
                log.info(f"No new laps since last poll. Total seen: {len(seen_laps)}")

            # ── Results / classification ───────────────────────────────
            all_results = extract_results_data(session)
            new_results = []
            for rec in all_results:
                key = rec.get("driver_code", "")
                if key not in seen_results:
                    seen_results.add(key)
                    new_results.append(rec)

            if new_results:
                output_data(new_results, "results")
                log.info(f"🏁 {len(new_results)} new result records pushed.")

        except FileNotFoundError:
            log.info("Data file not ready yet — waiting…")
        except Exception as e:
            log.error(f"Poll error: {e}", exc_info=True)


# ══════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════
def main():
    log.info("=" * 60)
    log.info(f"F1 Live Timing Pipeline — {YEAR} {GRAND_PRIX} {SESSION_TYPE}")
    log.info(f"Output mode: {'Event Hub → Stream Analytics → Power BI' if USE_EVENT_HUB else 'Direct Power BI push'}")
    log.info("=" * 60)

    # Validate auth
    if not os.environ.get("F1_USERNAME") or not os.environ.get("F1_PASSWORD"):
        log.warning(
            "⚠️  F1_USERNAME / F1_PASSWORD not set. SignalR auth will fail.\n"
            "   export F1_USERNAME='your@email.com'\n"
            "   export F1_PASSWORD='yourpassword'"
        )
    else:
        log.info(f"✅ F1 credentials found: {os.environ['F1_USERNAME']}")

    if USE_EVENT_HUB:
        log.info(f"✅ Event Hub configured: {EVENT_HUB_NAME}")
    elif "YOUR_PUSH_DATASET_URL" not in POWER_BI_PUSH_URL:
        log.info("✅ Power BI push URL configured.")
    else:
        log.warning("⚠️  Neither Power BI URL nor Event Hub configured — running in dry-run mode.")

    # Start poller in background thread
    poll_thread = threading.Thread(target=poll_and_push, daemon=True, name="Poller")
    poll_thread.start()
    log.info("✅ Poll thread started.")

    # Start recorder (blocks main thread — intentional)
    log.info("🚦 Starting live recording. Start 2–3 minutes BEFORE the session.")
    record_live_session()

    # Final flush after recording ends
    log.info("Recording finished. Final poll in progress…")
    time.sleep(POLL_INTERVAL + 5)
    log.info("✅ Pipeline complete.")


if __name__ == "__main__":
    main()